# Niederschlagsklassen – Modellvergleich, Start mit TiViT

Vergleichskandidaten: **TiViT**, **Timestral**, **Tiny Time Mixers**. Dieses Notebook implementiert zunächst TiViT.

Ziel ist die Niederschlagssumme eines zukünftigen dreistündigen Blocks: **0: 0 mm**, **1: >0 bis <3 mm**, **2: ≥3 mm**. Die Klassen sind Projektdefinitionen.

48 Stunden Historie mit denselben 20 Features wie in der vorhandenen Vergleichspipeline. Training vor 2022, Validation 2022–2023, retrospektive Evaluation 2024. Die Evaluation wurde bereits untersucht und ist kein unberührter Test.

Methode: Zeitfenster je Kanal in ein Graustufenbild umwandeln, eingefrorenen Vision Transformer anwenden, Kanalmerkmale konkatenieren und eine logistische Klassifikationsschicht nur auf Trainingsfällen fitten. Hier verwenden wir CLIP ViT-B/32, Layer 6, Segmentlänge 7, Schrittweite 1 und Mean-Pooling. Das ist eine kleinere TiViT-Adaption und keine Reproduktion der ViT-H-Ergebnisse.

Quellen: [TiViT Code](https://github.com/ExplainableML/TiViT), [Paper](https://arxiv.org/abs/2506.08641).


In [1]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'XAI-timeseries']
PROJECT_ROOT = next(p for p in candidates if (p / 'classification_data.py').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from classification_data import load_classification_bundle
from tivit_classifier import TiViTEncoder, time_images, run

DATA_DIR = PROJECT_ROOT / 'data/processed/precipitation_classification_v1'
HORIZON = 3
BATCH_SIZE = 8
# None = vollständiger Vergleich. Für technischen Probelauf z.B. 300 setzen.
MAX_CASES = None
DEVICE = 'cpu'  # alternativ 'mps' oder 'cuda', falls verfügbar
OUTPUT_DIR = PROJECT_ROOT / 'outputs/tivit_clip_b32' / ('full' if MAX_CASES is None else f'subset_{MAX_CASES}')
data = load_classification_bundle(DATA_DIR)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip, conda, or your preferred package management tool to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip, conda, or your preferred package management tool to install the fastparquet package.

## Daten und Klassenverteilung prüfen

Fehlende Inputs wurden bereits kausal nach der vorhandenen Pipeline behandelt. Keine globale Interpolation aus `final.parquet` verwenden. Klassenverteilungen bleiben unverändert.

In [ ]:
balance = []
for split in ['train', 'validation', 'evaluation']:
    x, y, cases = data.arrays(HORIZON, split)
    balance.append({'split': split, 'n': len(y), **{f'class{k}': int((y == k).sum()) for k in range(3)}})
    print(split, x.shape, cases.issue_time.min(), cases.issue_time.max())
display(pd.DataFrame(balance))


## Eingabebild ansehen

Ein Bild pro Eingabekanal; die Kanalmerkmale werden anschließend konkateniert. Konstante Kanäle werden als konstantes Bild dargestellt.

In [ ]:
x, _, _ = data.arrays(HORIZON, 'train')
rain_channel = data.features.columns.get_loc('R1')
image = time_images(x[:1, :, rain_channel:rain_channel+1])[0, 0].numpy()
plt.figure(figsize=(4, 4))
plt.imshow(image, cmap='gray', vmin=0, vmax=1)
plt.title('Historischer Niederschlag → TiViT-Bild')
plt.axis('off');


## Modell laden und Klassifikationsschicht trainieren

Benötigt die Pakete aus `requirements-tivit.txt`. Die CLIP-Gewichte liegen unter `models/clip-vit-base-patch32`; nach dem Klonen `git lfs pull` ausführen. Das Notebook lädt sie lokal im Safetensors-Format. Die Vision-Gewichte bleiben eingefroren. 20 Kanäle je Fall verursachen auch beim kleinen Backbone merklichen Rechenaufwand.

Validation und Evaluation dienen hier nur der Berichterstattung. Für spätere Auswahl von Layer, Backbone oder Regularisierung ausschließlich Validation verwenden; Evaluation erst nach Festlegung vergleichen.


In [ ]:
encoder = TiViTEncoder(model_id=str(PROJECT_ROOT / 'models/clip-vit-base-patch32'), device=DEVICE)
scores = run(DATA_DIR, OUTPUT_DIR, encoder, horizon=HORIZON,
             batch_size=BATCH_SIZE, max_cases=MAX_CASES)
display(scores)


## Auswertung

Macro-F1, Balanced Accuracy und Recall je Klasse berücksichtigen die seltenen Regenklassen. Der Multiclass-Brier-Score bewertet die drei Wahrscheinlichkeiten (kleiner ist besser).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, split in zip(axes, ['validation', 'evaluation']):
    matrix = np.loadtxt(OUTPUT_DIR / f'{split}_confusion.csv', delimiter=',').astype(int)
    ax.imshow(matrix, cmap='Blues')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center')
    ax.set(xticks=[0, 1, 2], yticks=[0, 1, 2], xlabel='Vorhergesagte Klasse', ylabel='Beobachtete Klasse', title=split)
plt.tight_layout()


## Weitere Modelle ergänzen

Timestral und Tiny Time Mixers müssen dieselben `case_id`s und Splits nutzen und Wahrscheinlichkeiten in der Reihenfolge `[0, 1, 2]` ausgeben. Numerische Forecasts benötigen dafür eine trainierte Klassifikationsschicht oder eine begründete Verteilungsumrechnung. Harte Schwellen auf einem Mittelwert liefern keine Klassenwahrscheinlichkeiten.

TiViT speichert `scores.csv`, Wahrscheinlichkeiten mit Fall-IDs, Confusion-Matrizen, die Klassifikationsparameter und `run.json`. Probeläufe auf Teilmengen separat halten. Scores anderer Modelle erst zusammenführen, wenn ihre Fall-IDs übereinstimmen.
